In [43]:
# numpy and pandas for data manipulation
import numpy as np
import pandas as pd 

# File system manangement
import os


# ------------------- IMPORT SRC ------------------------------------
# src is the parent folder of notebooks, so we need to add it to sys.path to import config and utils
import sys
notebook_dir = os.getcwd() 

# Parent folder of src
project_root = os.path.abspath(os.path.join(notebook_dir, "..")) 
sys.path.append(project_root)

print("sys.path contains:", sys.path[-1])

from src.config import Config as Config  
from src.data_loader import load_data, prepare_data

cfg = Config

KAGGLE_EVAL = cfg.KAGGLE_EVAL
RANDOM_STATE = cfg.RANDOM_STATE
TASK = cfg.TASK
USE_POSTPROCESSING = cfg.USE_POSTPROCESSING
TARGET = cfg.TARGET

# -------------------------------------------------------


from sklearn.model_selection import train_test_split

# Suppress warnings 
import warnings
warnings.filterwarnings('ignore')


# matplotlib and seaborn for plotting
import matplotlib.pyplot as plt
import seaborn as sns

from sklearnex import patch_sklearn
patch_sklearn()  # patches scikit-learn algorithms
# from sklearnex import unpatch_sklearn
# unpatch_sklearn()




X_train, X_test, y_train, y_test = load_data("encoded", data_dir='./data/')




# Prepare
X_train, X_test, y_train_numeric, y_test_numeric, test_ids, num_classes, int_to_label = prepare_data(
    X_train, X_test, y_train, y_test, target=cfg.TARGET, drop_id=True # ,label_map = {"A": 1, "B": 0, "C": 2}
)

cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
















sys.path contains: /home/ismail


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


Number of classes: 2
X_train shape: (80024, 267)
X_test shape: (34297, 267)
y_train shape: (80024,)
y_test shape: (34297,)


In [44]:





import warnings, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.decomposition   import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model    import LogisticRegression
from sklearn.metrics         import (roc_auc_score, classification_report,
                                     confusion_matrix, f1_score)
from sklearn.ensemble        import RandomForestClassifier

import gymnasium as gym
from gymnasium import spaces

OUT = Path("./outputs")
OUT.mkdir(exist_ok=True)

SEED = 42
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

print("=" * 65)
print("LOADING & PREPROCESSING DATA")
print("=" * 65)

df_raw = pd.read_csv("./sample_data.csv")
print(f"Raw shape: {df_raw.shape}")

# ── (a) binary flags from EDA / CDF analysis ─────────────────────────────────
df = df_raw.copy()

df["has_prior_denial"]         = (df["prior_denial_indicator"].fillna(0) > 0).astype(int)
df["has_customer_value"]       = (df["customer_value_score"]             > 0).astype(int)
df["is_zero_touchpoints"]      = (df["communication_touchpoints"]       == 0).astype(int)
df["coverage_limit_high"]      = (df["coverage_limit"]                  > 12.5).astype(int)
df["policy_premium_high"]      = (df["policy_premium_annual"].fillna(df["policy_premium_annual"].median()) > 8).astype(int)
df["recovery_is_one"]          = (df["recovery_probability"]            == 1).astype(int)
df["is_renewal_month_7"]       = (df["policy_renewal_months"].fillna(0) == 7).astype(int)
df["cross_sell_high"]          = (df["cross_sell_index"].fillna(0)       > 15).astype(int)
df["is_fast_claim"]            = (df["claim_processing_days"].fillna(0)  < 1).astype(int)
df["is_high_urban_density"]    = (df["urban_density_index"].fillna(0)    > 5.5).astype(int)
df["is_nonzero_severity"]      = (df["severity_index"]                  > 0).astype(int)
df["denial_x_processing"]      = df["has_prior_denial"] * df["claim_processing_days"].fillna(0)

# ── (b) categorical encoding (OHE for low-card, frequency for claim_detail_code) ──
cat_ohe = [
    "claim_type","geographic_region","claim_channel",
    "fraud_indicator_flag","provider_network_status",
    "document_completeness","priority_level","risk_category",
    "customer_segment","payment_method","automation_eligible",
    "adjuster_recommendation","claim_complexity","policy_status",
    "coverage_type","policy_type","claim_source","location_code"
]
cat_ohe_present = [c for c in cat_ohe if c in df.columns]
df[cat_ohe_present] = df[cat_ohe_present].fillna("MISSING")
df = pd.get_dummies(df, columns=cat_ohe_present, drop_first=True)

# claim_detail_code → frequency encode
if "claim_detail_code" in df.columns:
    freq = df["claim_detail_code"].value_counts() / len(df)
    df["claim_detail_code_freq"] = df["claim_detail_code"].map(freq).fillna(0)
    df.drop(columns=["claim_detail_code"], inplace=True)

# ── (c) drop noisy / near-zero-MI features  ──────────────────────────────────
drop_cols = [
    "ID","composite_risk_index","deductible_amount","data_completeness_score",
    "depreciation_rate","customer_satisfaction_score","follow_up_responsiveness_score",
    "payment_timeliness_score","policy_modifications_count","medication_count",
    "claim_to_premium_ratio","endorsement_value_ratio","duplicate_claim_similarity",
    "straight_through_processing_score","days_since_policy_start",
    # high-correlation duplicate (r=0.998 pair: keep witnesses_count, drop api_response_quality)
    "api_response_quality",
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# ── (d) target, split ────────────────────────────────────────────────────────
y = df["target"].values
df.drop(columns=["target"], inplace=True)

df = df.select_dtypes(include=[np.number])
df = df.astype(float)

df.fillna(df.median(), inplace=True)

X = df.values
print(f"Feature matrix: {X.shape}")
print(f"Class distribution  0={np.sum(y==0)} ({100*np.mean(y==0):.1f}%)  "
      f"1={np.sum(y==1)} ({100*np.mean(y==1):.1f}%)")

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

K_PCA = 15
pca = PCA(n_components=K_PCA, random_state=SEED)
X_tr_pca = pca.fit_transform(X_tr_sc)
X_te_pca = pca.transform(X_te_sc)
print(f"PCA k={K_PCA} explains {pca.explained_variance_ratio_.sum()*100:.1f}% variance")

# ── (f) also build a rich supervised signal for reward shaping ───────────────
rf_sup = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced", random_state=SEED, n_jobs=-1)
rf_sup.fit(X_tr_sc, y_tr)
p_tr_sup = rf_sup.predict_proba(X_tr_sc)[:,1]
p_te_sup = rf_sup.predict_proba(X_te_sc)[:,1]
print(f"Supervised RF AUC (test): {roc_auc_score(y_te, p_te_sup):.4f}")


print("\n" + "=" * 65)
print("Q12 — CLAIMS ADJUDICATION ENVIRONMENT")
print("=" * 65)


class ClaimsEnv(gym.Env):

    metadata = {"render_modes": []}

    # reward constants (Q12 justification above)
    R_CORRECT_APPROVE   = +10.0
    R_CORRECT_DENY      = +8.0   # slightly lower: deny has higher uncertainty
    R_WRONG_FAST_TRACK  = -20.0  # critical error
    R_WRONG_DENY        = -3.0   # customer dissatisfaction
    R_DELAY             = -1.0   # doc request or review
    R_RISK_SCALE        = -5.0   # supervised shaping coefficient

    def __init__(self, X_pca, X_flags, y, sup_probs, mode="train", seed=42):
        super().__init__()
        self.X_pca    = X_pca.astype(np.float32)
        self.X_flags  = X_flags.astype(np.float32)
        self.y        = y.astype(int)
        self.sup_probs = sup_probs.astype(np.float32)
        self.mode     = mode
        self.n        = len(y)
        self.rng      = np.random.default_rng(seed)
        self._idx     = 0

        n_obs = X_pca.shape[1] + X_flags.shape[1]   # 15 + 5 = 20
        # observation: continuous 20-dim vector in [-inf, +inf]
        self.observation_space = spaces.Box(
            low  = -np.inf,
            high =  np.inf,
            shape = (n_obs,),
            dtype = np.float32
        )
        # 4 discrete actions
        self.action_space = spaces.Discrete(4)
        self.action_names = {
            0: "Fast-track Approve",
            1: "Request Docs",
            2: "Manual Review",
            3: "Deny"
        }

    def _get_obs(self, idx):
        return np.concatenate([self.X_pca[idx], self.X_flags[idx]])

    def reset(self, seed=None, options=None):
        if self.mode == "train":
            self._current_idx = int(self.rng.integers(0, self.n))
        else:
            self._current_idx = self._idx % self.n
            self._idx += 1
        obs = self._get_obs(self._current_idx)
        return obs, {}

    def step(self, action):
        idx   = self._current_idx
        label = self.y[idx]
        sp    = float(self.sup_probs[idx])     # P(class=1) from supervised model

        if action == 0:   # fast-track approve
            if label == 1:
                reward = self.R_CORRECT_APPROVE
            else:
                reward = self.R_WRONG_FAST_TRACK + self.R_RISK_SCALE * (1.0 - sp)

        elif action == 3:  # deny
            if label == 0:
                reward = self.R_CORRECT_DENY
            else:
                reward = self.R_WRONG_DENY

        else:              # action 1 (docs) or 2 (review)
            # delay cost always applies
            reward = self.R_DELAY
            # bonus: if supervised model is uncertain (0.3 < sp < 0.7), routing
            # to review is the right call → small bonus to break flat reward
            uncertainty = 1.0 - abs(2*sp - 1.0)    # 0=certain, 1=totally uncertain
            if action == 2:  # manual review bonus
                reward += 1.5 * uncertainty
            elif action == 1: # docs request bonus
                reward += 0.8 * uncertainty

        terminated = True   # every episode is a single claim decision
        truncated  = False
        info = {
            "label": label, "action": action,
            "sup_prob": sp, "reward": reward,
            "action_name": self.action_names[action]
        }
        return self._get_obs(idx), reward, terminated, truncated, info


FLAG_COLS = [
    "has_prior_denial","has_customer_value","is_fast_claim",
    "coverage_limit_high","is_zero_touchpoints"
]
flag_idx = [list(df.columns).index(c) for c in FLAG_COLS if c in df.columns]
X_tr_flags = X_tr[:, flag_idx]
X_te_flags = X_te[:, flag_idx]

# ensure 5 flags
if X_tr_flags.shape[1] < 5:
    pad = np.zeros((X_tr_flags.shape[0], 5 - X_tr_flags.shape[1]))
    X_tr_flags = np.hstack([X_tr_flags, pad])
    X_te_flags = np.hstack([X_te_flags, pad])

train_env = ClaimsEnv(X_tr_pca, X_tr_flags, y_tr, p_tr_sup, mode="train", seed=SEED)
test_env  = ClaimsEnv(X_te_pca, X_te_flags, y_te, p_te_sup, mode="test",  seed=SEED+1)

print("Environment created.")
print(f"  Obs space:    {train_env.observation_space}")
print(f"  Action space: {train_env.action_space}")
print(f"  Train claims: {train_env.n}  |  Test claims: {test_env.n}")
print()
print("REWARD FUNCTION SUMMARY")
print(f"  Correct fast-track approve : {ClaimsEnv.R_CORRECT_APPROVE:+.0f}")
print(f"  Correct deny               : {ClaimsEnv.R_CORRECT_DENY:+.0f}")
print(f"  Wrong fast-track (risky)   : {ClaimsEnv.R_WRONG_FAST_TRACK:+.0f}  ← large penalty")
print(f"  Wrong deny (legit claim)   : {ClaimsEnv.R_WRONG_DENY:+.0f}")
print(f"  Delay cost (docs/review)   : {ClaimsEnv.R_DELAY:+.0f}")
print(f"  Risk shaping coefficient   : {ClaimsEnv.R_RISK_SCALE:+.0f}  ← addresses flat curve")


# ══════════════════════════════════════════════════════════════════════════════
# Q13 — TABULAR Q-LEARNING WITH DISCRETISED STATE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("Q13 — Q-LEARNING AGENT")
print("=" * 65)


LOADING & PREPROCESSING DATA
Raw shape: (14767, 133)
Feature matrix: (14767, 110)
Class distribution  0=3526 (23.9%)  1=11241 (76.1%)
PCA k=15 explains 78.0% variance
Supervised RF AUC (test): 0.6845

Q12 — CLAIMS ADJUDICATION ENVIRONMENT
Environment created.
  Obs space:    Box(-inf, inf, (20,), float32)
  Action space: Discrete(4)
  Train claims: 11813  |  Test claims: 2954

REWARD FUNCTION SUMMARY
  Correct fast-track approve : +10
  Correct deny               : +8
  Wrong fast-track (risky)   : -20  ← large penalty
  Wrong deny (legit claim)   : -3
  Delay cost (docs/review)   : -1
  Risk shaping coefficient   : -5  ← addresses flat curve

Q13 — Q-LEARNING AGENT


In [45]:


BINS         = 5     # quantisation bins per PCA dimension
N_EPISODES   = 80_000
ALPHA_START  = 0.35
ALPHA_MIN    = 0.05
GAMMA        = 0.1
EPS_START    = 1.0
EPS_END      = 0.05
DECAY_STEPS  = 50_000
EVAL_EVERY   = 500   # episodes between logging
EVAL_WINDOW  = 300   # rolling window for smoothing

obs_dim = train_env.observation_space.shape[0]
ALL_TRAIN_OBS = np.hstack([X_tr_pca, X_tr_flags]).astype(np.float32)
bin_edges = []
for d in range(obs_dim):
    col = ALL_TRAIN_OBS[:, d]
    edges = np.quantile(col, np.linspace(0, 1, BINS + 1))
    edges[0]  -= 1e-8    # include minimum
    edges[-1] += 1e-8    # include maximum
    bin_edges.append(edges)

def discretise(obs):
    """Map continuous obs vector → integer state index (via mixed-radix)."""
    digits = []
    for d in range(obs_dim):
        b = int(np.digitize(obs[d], bin_edges[d]) - 1)
        b = max(0, min(b, BINS - 1))
        digits.append(b)
    # pack into single int (base-BINS mixed radix)
    idx = 0
    for b in digits:
        idx = idx * BINS + b
    return idx

Q = {}

def get_Q(s):
    if s not in Q:
        Q[s] = np.zeros(4, dtype=np.float32)
    return Q[s]

def q_policy(obs, eps):
    """ε-greedy action selection."""
    if np.random.random() < eps:
        return train_env.action_space.sample()
    s  = discretise(obs)
    return int(np.argmax(get_Q(s)))

# ── training loop ─────────────────────────────────────────────────────────────
print(f"Training for {N_EPISODES:,} episodes …")
t0 = time.time()

episode_rewards  = []
rolling_rewards  = []
episode_acc      = []    # was the decision "correct"?
log_steps        = []
q_table_sizes    = []

for ep in range(1, N_EPISODES + 1):
    frac   = min(ep / DECAY_STEPS, 1.0)
    eps    = EPS_START + frac * (EPS_END - EPS_START)
    alpha  = ALPHA_MIN + 0.5 * (ALPHA_START - ALPHA_MIN) * (1 + np.cos(np.pi * frac))

    obs, _ = train_env.reset()
    action = q_policy(obs, eps)
    obs2, reward, terminated, truncated, info = train_env.step(action)

    s  = discretise(obs)
    s2 = discretise(obs2)
    q_arr = get_Q(s)
    best_next = float(np.max(get_Q(s2))) if not terminated else 0.0
    q_arr[action] += alpha * (reward + GAMMA * best_next - q_arr[action])

    episode_rewards.append(reward)
    label  = info["label"]
    # "correct": fast-track→label=1  OR  deny→label=0
    correct = int((action==0 and label==1) or (action==3 and label==0))
    episode_acc.append(correct)

    if ep % EVAL_EVERY == 0:
        window = episode_rewards[-EVAL_WINDOW:]
        rw = np.mean(window)
        rolling_rewards.append(rw)
        log_steps.append(ep)
        q_table_sizes.append(len(Q))

        if ep % 10_000 == 0:
            acc_w = np.mean(episode_acc[-EVAL_WINDOW:])
            print(f"  ep {ep:>6,}  ε={eps:.3f}  α={alpha:.4f}  "
                  f"avg_reward={rw:+.3f}  acc={acc_w:.3f}  |Q|={len(Q):,}")

print(f"Training done in {time.time()-t0:.1f}s  |  Q-table size: {len(Q):,} states")


def evaluate(env, n_steps=None, eps=0.0):
    if n_steps is None:
        n_steps = env.n
    records = []
    for _ in range(n_steps):
        obs, _ = env.reset()
        s = discretise(obs)
        action = int(np.argmax(get_Q(s))) if eps == 0 else env.action_space.sample()
        _, reward, _, _, info = env.step(action)
        records.append({
            "reward": reward,
            "label":  info["label"],
            "action": info["action"],
            "sup_prob": info["sup_prob"],
        })
    return pd.DataFrame(records)

results_rl = evaluate(test_env, eps=0.0)
mean_r = results_rl["reward"].mean()
print(f"\nRL agent test  avg_reward = {mean_r:+.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# Q13 PLOT — Learning curve
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Q13 — Q-Learning Agent: Training Dynamics", fontsize=14, fontweight="bold")

# (A) Rolling reward
ax = axes[0,0]
ax.plot(log_steps, rolling_rewards, lw=1.8, color="#1f77b4")
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("Training Episode")
ax.set_ylabel(f"Avg Reward (window={EVAL_WINDOW})")
ax.set_title("(A) Rolling Average Reward")
ax.grid(alpha=0.3)

# (B) ε and α decay
ax = axes[0,1]
eps_curve = [EPS_START + min(e/DECAY_STEPS,1.0)*(EPS_END-EPS_START)
             for e in range(0, N_EPISODES+1, EVAL_EVERY)]
alpha_curve = [ALPHA_MIN + 0.5*(ALPHA_START-ALPHA_MIN)*(1+np.cos(np.pi*min(e/DECAY_STEPS,1.0)))
               for e in range(0, N_EPISODES+1, EVAL_EVERY)]
ax2 = ax.twinx()
ax.plot(range(0, N_EPISODES+1, EVAL_EVERY), eps_curve,   color="crimson",  lw=1.5, label="ε (explore)")
ax2.plot(range(0, N_EPISODES+1, EVAL_EVERY), alpha_curve, color="darkorange", lw=1.5, ls="--", label="α (lr)")
ax.set_xlabel("Episode"); ax.set_ylabel("ε", color="crimson")
ax2.set_ylabel("α", color="darkorange")
ax.set_title("(B) Exploration & Learning-Rate Schedules")
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, loc="upper right", fontsize=8)
ax.grid(alpha=0.3)

# (C) Q-table growth
ax = axes[1,0]
ax.plot(log_steps, q_table_sizes, color="seagreen", lw=1.5)
ax.set_xlabel("Episode"); ax.set_ylabel("Unique States Visited")
ax.set_title("(C) Q-Table Growth")
ax.grid(alpha=0.3)

# (D) Action distribution over training (sampled every 500)
ax = axes[1,1]
# replay a smaller chunk to get action distribution
ep_sample = min(5000, N_EPISODES)
# use last 10k episodes to show converged policy
last_actions = []
for _ in range(2000):
    obs, _ = train_env.reset()
    s = discretise(obs)
    q_arr = get_Q(s)
    # ε=0.1 near-greedy
    if np.random.random() < 0.1:
        a = train_env.action_space.sample()
    else:
        a = int(np.argmax(q_arr))
    last_actions.append(a)
vals, cnts = np.unique(last_actions, return_counts=True)
colors = ["#2ca02c","#ff7f0e","#9467bd","#d62728"]
bar_labels = ["0: Fast-track","1: Req. Docs","2: Review","3: Deny"]
ax.bar([bar_labels[v] for v in vals], cnts/len(last_actions)*100,
       color=[colors[v] for v in vals], edgecolor="black", linewidth=0.5)
ax.set_ylabel("% of Decisions")
ax.set_title("(D) Converged Policy Action Distribution")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / "Q13_learning_curve.png", dpi=160, bbox_inches="tight")
plt.close()
print("Saved Q13_learning_curve.png")


# ══════════════════════════════════════════════════════════════════════════════
# Q14 — BASELINE POLICIES + COMPARISON
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("Q14 — EVALUATION vs BASELINES")
print("=" * 65)

def policy_always_approve(obs, env): return 0
def policy_always_review (obs, env): return 2
def policy_random        (obs, env): return env.action_space.sample()

def policy_supervised(obs, env):
    """Use RF prob to mimic a supervised approach with a threshold."""
    idx = env._current_idx
    sp  = env.sup_probs[idx]
    if   sp >= 0.80: return 0   # high confidence approve
    elif sp <= 0.30: return 3   # high confidence deny
    elif sp <= 0.55: return 2   # uncertain → review
    else:            return 1   # moderate → request docs

def run_policy(env, policy_fn, label=""):
    """Evaluate a policy function over the full test set."""
    records = []
    for _ in range(env.n):
        obs, _ = env.reset()
        a = policy_fn(obs, env)
        _, r, _, _, info = env.step(a)
        records.append({
            "reward": r, "label": info["label"],
            "action": a, "sup_prob": info["sup_prob"]
        })
    df_r = pd.DataFrame(records)
    # compute classification-style metrics
    # treat action=0 as "approve" (predicted=1), action=3 as "deny" (predicted=0)
    # actions 1,2 → defer (we call this "uncertain"; map to predicted=1 for AUC)
    pred_label = (df_r["action"] != 3).astype(int)   # 0=deny, 1=not-deny
    try:
        auc = roc_auc_score(df_r["label"], pred_label)
    except Exception:
        auc = np.nan
    f1_0 = f1_score(df_r["label"], pred_label, pos_label=0, zero_division=0)
    f1_1 = f1_score(df_r["label"], pred_label, pos_label=1, zero_division=0)
    acc  = (df_r["label"] == pred_label).mean()
    # error rates
    wrong_ft = ((df_r["action"]==0) & (df_r["label"]==0)).sum()   # risky fast-track
    wrong_dn = ((df_r["action"]==3) & (df_r["label"]==1)).sum()   # wrong deny
    return {
        "policy": label,
        "avg_reward":     df_r["reward"].mean(),
        "total_reward":   df_r["reward"].sum(),
        "accuracy":       acc,
        "auc":            auc,
        "f1_class0":      f1_0,
        "f1_class1":      f1_1,
        "wrong_fasttrack":wrong_ft,
        "wrong_deny":     wrong_dn,
        "action_dist":    df_r["action"].value_counts().to_dict(),
        "_df":            df_r,
    }

# re-init test env for each policy (sequential mode)
def fresh_test_env():
    env = ClaimsEnv(X_te_pca, X_te_flags, y_te, p_te_sup, mode="test", seed=SEED+1)
    env._idx = 0
    return env

results = {}

for name, fn in [
    ("Random",           policy_random),
    ("Always Approve",   policy_always_approve),
    ("Always Review",    policy_always_review),
    ("Supervised RF",    policy_supervised),
]:
    e = fresh_test_env()
    results[name] = run_policy(e, fn, label=name)
    print(f"  {name:<20} avg_r={results[name]['avg_reward']:+.3f}  "
          f"acc={results[name]['accuracy']:.3f}  auc={results[name]['auc']:.3f}")

# RL agent
e = fresh_test_env()
def policy_rl(obs, env):
    s = discretise(obs)
    return int(np.argmax(get_Q(s)))
results["Q-Learning"] = run_policy(e, policy_rl, label="Q-Learning")
r_rl = results["Q-Learning"]
print(f"  {'Q-Learning':<20} avg_r={r_rl['avg_reward']:+.3f}  "
      f"acc={r_rl['accuracy']:.3f}  auc={r_rl['auc']:.3f}")


# ── Summary table ─────────────────────────────────────────────────────────────
summary = []
for k, v in results.items():
    row = {c: v[c] for c in
           ["policy","avg_reward","accuracy","auc","f1_class0","f1_class1",
            "wrong_fasttrack","wrong_deny"]}
    summary.append(row)
df_sum = pd.DataFrame(summary).set_index("policy")
print("\n── Full Comparison Table ──")
print(df_sum.round(4).to_string())


# ══════════════════════════════════════════════════════════════════════════════
# Q14 PLOTS
# ══════════════════════════════════════════════════════════════════════════════
POLICY_ORDER = ["Random","Always Approve","Always Review","Supervised RF","Q-Learning"]
PAL = {
    "Random":        "#aec6cf",
    "Always Approve":"#ffb347",
    "Always Review": "#c8a2c8",
    "Supervised RF": "#77dd77",
    "Q-Learning":    "#1f77b4",
}
action_colors = ["#2ca02c","#ff7f0e","#9467bd","#d62728"]
action_labels = ["0: Approve","1: Req. Docs","2: Review","3: Deny"]

# ── Figure 1: Overall metric comparison ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Q14 — Policy Comparison on Test Set", fontsize=14, fontweight="bold")

metrics = [
    ("avg_reward",     "Average Reward per Claim"),
    ("accuracy",       "Accuracy"),
    ("auc",            "ROC-AUC"),
    ("f1_class0",      "F1 Score — Class 0 (Deny)"),
    ("f1_class1",      "F1 Score — Class 1 (Approve)"),
    ("wrong_fasttrack","# Risky Claims Fast-Tracked"),
]

for ax, (metric, title) in zip(axes.flat, metrics):
    vals  = [df_sum.loc[p, metric] for p in POLICY_ORDER]
    cols  = [PAL[p] for p in POLICY_ORDER]
    bars  = ax.bar(POLICY_ORDER, vals, color=cols, edgecolor="black", lw=0.6)
    ax.set_title(title, fontsize=9)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
    ax.grid(axis="y", alpha=0.3)
    # highlight RL bar
    idx_rl = POLICY_ORDER.index("Q-Learning")
    bars[idx_rl].set_edgecolor("red"); bars[idx_rl].set_linewidth(2)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+abs(bar.get_height())*0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.savefig(OUT / "Q14_policy_comparison.png", dpi=160, bbox_inches="tight")
plt.close()
print("\nSaved Q14_policy_comparison.png")

# ── Figure 2: Action distribution per policy ─────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
fig.suptitle("Q14 — Action Distribution by Policy", fontsize=12, fontweight="bold")

for ax, pname in zip(axes, POLICY_ORDER):
    dist = results[pname]["action_dist"]
    total = sum(dist.values())
    fracs = [dist.get(a, 0)/total*100 for a in range(4)]
    ax.bar(range(4), fracs, color=action_colors, edgecolor="black", lw=0.5)
    ax.set_xticks(range(4))
    ax.set_xticklabels(["Approve","Docs","Review","Deny"], rotation=45, ha="right", fontsize=7)
    ax.set_title(pname, fontsize=8)
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.3)
    for i, f in enumerate(fracs):
        if f > 2:
            ax.text(i, f+1, f"{f:.0f}%", ha="center", fontsize=6)
axes[0].set_ylabel("% of Decisions")

plt.tight_layout()
plt.savefig(OUT / "Q14_action_distribution.png", dpi=160, bbox_inches="tight")
plt.close()
print("Saved Q14_action_distribution.png")

# ── Figure 3: RL decision pattern — heatmap over PC1/PC2 & supervised prob ───
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Q14 — RL Agent Decision Patterns", fontsize=13, fontweight="bold")

df_rl = results["Q-Learning"]["_df"].copy()
df_rl["pc1"] = X_te_pca[:,0]
df_rl["pc2"] = X_te_pca[:,1]

# (A) Action scatter on PC1/PC2
ax = axes[0]
for a in range(4):
    mask = df_rl["action"] == a
    ax.scatter(df_rl.loc[mask,"pc1"], df_rl.loc[mask,"pc2"],
               c=action_colors[a], alpha=0.25, s=4, label=action_labels[a])
ax.set_xlabel("PC1 (66.95% var)"); ax.set_ylabel("PC2 (6.73% var)")
ax.set_title("(A) Decisions in PCA Space")
ax.legend(markerscale=2.5, fontsize=7)
ax.grid(alpha=0.2)

# (B) Action vs supervised probability
ax = axes[1]
bp_data = [df_rl.loc[df_rl["action"]==a,"sup_prob"].values for a in range(4)]
bp = ax.boxplot(bp_data, patch_artist=True, notch=False,
                medianprops=dict(color="black",lw=2))
for patch, c in zip(bp["boxes"], action_colors):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_xticklabels(["Approve","Docs","Review","Deny"])
ax.set_ylabel("Supervised Probability (P(class=1))")
ax.set_title("(B) Sup. Prob by RL Action")
ax.grid(axis="y", alpha=0.3)

# (C) Confusion matrix for RL policy
ax = axes[2]
pred_rl = (df_rl["action"] != 3).astype(int)
cm = confusion_matrix(df_rl["label"], pred_rl)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Pred: Deny","Pred: Not-Deny"],
            yticklabels=["True: 0 (Deny)","True: 1 (Approve)"],
            linewidths=0.5, linecolor="gray")
ax.set_title("(C) RL Agent Confusion Matrix")

plt.tight_layout()
plt.savefig(OUT / "Q14_rl_decision_patterns.png", dpi=160, bbox_inches="tight")
plt.close()
print("Saved Q14_rl_decision_patterns.png")


Training for 80,000 episodes …
  ep 10,000  ε=0.810  α=0.3214  avg_reward=+0.367  acc=0.307  |Q|=5,420
  ep 20,000  ε=0.620  α=0.2464  avg_reward=+2.664  acc=0.447  |Q|=7,506
  ep 30,000  ε=0.430  α=0.1536  avg_reward=+2.985  acc=0.487  |Q|=8,370
  ep 40,000  ε=0.240  α=0.0786  avg_reward=+5.128  acc=0.617  |Q|=8,762
  ep 50,000  ε=0.050  α=0.0500  avg_reward=+6.826  acc=0.743  |Q|=8,927
  ep 60,000  ε=0.050  α=0.0500  avg_reward=+6.268  acc=0.717  |Q|=8,994
  ep 70,000  ε=0.050  α=0.0500  avg_reward=+6.547  acc=0.713  |Q|=9,016
  ep 80,000  ε=0.050  α=0.0500  avg_reward=+6.743  acc=0.720  |Q|=9,025
Training done in 21.2s  |  Q-table size: 9,025 states

RL agent test  avg_reward = +2.2581
Saved Q13_learning_curve.png

Q14 — EVALUATION vs BASELINES
  Random               avg_r=+0.202  acc=0.625  auc=0.496
  Always Approve       avg_r=+2.177  acc=0.761  auc=0.500
  Always Review        avg_r=+0.123  acc=0.761  auc=0.500
  Supervised RF        avg_r=+0.815  acc=0.764  auc=0.547
  Q-Learni

In [46]:


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    "Q14 — Flat-Reward Diagnosis & Solution",
    fontsize=12, fontweight="bold"
)

# (A) Learning curve with phases annotated
ax = axes[0]
ax.plot(log_steps, rolling_rewards, lw=1.5, color="#1f77b4")
ax.axhline(0, color="gray", lw=0.8, ls=":")
# Find approximate inflection
mid_ep = DECAY_STEPS
ax.axvline(mid_ep, color="red", lw=1, ls="--", label="ε decay end")
ax.fill_between(log_steps, rolling_rewards,
                alpha=0.15, color="#1f77b4")
ax.set_xlabel("Episode"); ax.set_ylabel("Rolling Avg Reward")
ax.set_title("(A) Learning Curve — RL Agent")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
# annotate phases
y_top = max(rolling_rewards) * 1.1
ax.annotate("Exploration\n(high ε)", xy=(5000, y_top*0.5),
            fontsize=8, color="red", ha="center")
ax.annotate("Exploitation\n(low ε)", xy=(70000, y_top*0.5),
            fontsize=8, color="navy", ha="center")

# (B) Reward distribution comparison — with vs without shaping
# Simulate a no-shaping reward just using binary correct/wrong
ax = axes[1]
noshape_r = []
shape_r   = df_rl["reward"].tolist()
for _, row in df_rl.iterrows():
    if row["action"]==0 and row["label"]==1:
        noshape_r.append(10.0)
    elif row["action"]==0 and row["label"]==0:
        noshape_r.append(-20.0)   # no shaping: flat penalty
    elif row["action"]==3 and row["label"]==0:
        noshape_r.append(8.0)
    elif row["action"]==3 and row["label"]==1:
        noshape_r.append(-3.0)
    else:
        noshape_r.append(-1.0)    # no uncertainty bonus

bins = np.linspace(-25, 15, 40)
ax.hist(noshape_r, bins=bins, alpha=0.5, label="Without shaping", color="tomato",  density=True)
ax.hist(shape_r,   bins=bins, alpha=0.5, label="With shaping",    color="#1f77b4", density=True)
ax.set_xlabel("Reward"); ax.set_ylabel("Density")
ax.set_title("(B) Reward Distribution:\nShaping vs No Shaping")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (C) Per-action average reward by class (shows agent has learned differential)
ax = axes[2]
grouped = df_rl.groupby(["action","label"])["reward"].mean().unstack(fill_value=0)
grouped.index = [action_labels[i] for i in grouped.index]
grouped.plot(kind="bar", ax=ax, color=["#ff6b6b","#4ecdc4"],
             edgecolor="black", linewidth=0.5, width=0.7)
ax.set_xlabel("")
ax.set_ylabel("Average Reward")
ax.set_title("(C) Avg Reward by Action × True Label\n(agent differentiated correctly)")
ax.legend(["Class 0 (Deny-correct)","Class 1 (Approve-correct)"], fontsize=7)
ax.tick_params(axis="x", rotation=30)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / "Q14_flat_reward_solution.png", dpi=160, bbox_inches="tight")
plt.close()
print("Saved Q14_flat_reward_solution.png")


# ── Figure 5: Full dashboard / final summary ─────────────────────────────────
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.40)
fig.suptitle("Part III Summary Dashboard — RL vs Supervised vs Baselines",
             fontsize=14, fontweight="bold")

# Row 0 Col 0-1: reward bar
ax = fig.add_subplot(gs[0, 0:2])
vals = [df_sum.loc[p,"avg_reward"] for p in POLICY_ORDER]
bars = ax.bar(POLICY_ORDER, vals, color=[PAL[p] for p in POLICY_ORDER],
              edgecolor="black", lw=0.7)
bars[POLICY_ORDER.index("Q-Learning")].set_edgecolor("red")
bars[POLICY_ORDER.index("Q-Learning")].set_linewidth(2.5)
ax.set_ylabel("Avg Reward"); ax.set_title("Average Reward per Claim")
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.tick_params(axis="x", rotation=30, labelsize=8)
ax.grid(axis="y", alpha=0.3)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height() + (0.05 if v >= 0 else -0.3),
            f"{v:.2f}", ha="center", fontsize=7)

# Row 0 Col 2-3: Risky fast-tracks
ax = fig.add_subplot(gs[0, 2:4])
vals2 = [df_sum.loc[p,"wrong_fasttrack"] for p in POLICY_ORDER]
bars2 = ax.bar(POLICY_ORDER, vals2, color=[PAL[p] for p in POLICY_ORDER],
               edgecolor="black", lw=0.7)
ax.set_ylabel("Count"); ax.set_title("# Risky Claims Wrongly Fast-Tracked\n(lower = safer)")
ax.tick_params(axis="x", rotation=30, labelsize=8)
ax.grid(axis="y", alpha=0.3)
for bar, v in zip(bars2, vals2):
    ax.text(bar.get_x()+bar.get_width()/2, v+2, str(int(v)), ha="center", fontsize=7)

# Row 1 Col 0: Learning curve
ax = fig.add_subplot(gs[1, 0:2])
ax.plot(log_steps, rolling_rewards, lw=1.5, color="#1f77b4")
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("Episode"); ax.set_ylabel("Rolling Reward")
ax.set_title("RL Learning Curve")
ax.grid(alpha=0.3)

# Row 1 Col 2: Action dist RL
ax = fig.add_subplot(gs[1, 2])
dist_rl = results["Q-Learning"]["action_dist"]
tot = sum(dist_rl.values())
fracs = [dist_rl.get(a,0)/tot*100 for a in range(4)]
ax.bar(range(4), fracs, color=action_colors, edgecolor="black", lw=0.5)
ax.set_xticks(range(4)); ax.set_xticklabels(["Appr","Docs","Rev","Deny"], fontsize=7)
ax.set_ylabel("%"); ax.set_title("RL Action Distribution"); ax.grid(axis="y", alpha=0.3)

# Row 1 Col 3: F1 class 0
ax = fig.add_subplot(gs[1, 3])
f1s = [df_sum.loc[p,"f1_class0"] for p in POLICY_ORDER]
ax.barh(POLICY_ORDER, f1s, color=[PAL[p] for p in POLICY_ORDER], edgecolor="black", lw=0.5)
ax.set_xlabel("F1 Score"); ax.set_title("F1 — Class 0 (Deny)\nMinority class")
ax.axvline(df_sum.loc["Q-Learning","f1_class0"], color="red", lw=1.5, ls="--")
ax.grid(axis="x", alpha=0.3)

# Row 2: Confusion matrices side by side (RL vs Supervised)
for col_idx, pname in enumerate(["Q-Learning","Supervised RF","Always Approve","Random"]):
    ax = fig.add_subplot(gs[2, col_idx])
    df_p = results[pname]["_df"]
    pred = (df_p["action"] != 3).astype(int)
    cm   = confusion_matrix(df_p["label"], pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["P:0","P:1"], yticklabels=["T:0","T:1"],
                linewidths=0.3, annot_kws={"size":8})
    ax.set_title(pname, fontsize=8)

plt.savefig(OUT / "Q14_summary_dashboard.png", dpi=160, bbox_inches="tight")
plt.close()
print("Saved Q14_summary_dashboard.png")


# ══════════════════════════════════════════════════════════════════════════════
# PRINT FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("FINAL RESULTS SUMMARY")
print("=" * 65)
print(df_sum[["avg_reward","accuracy","auc","f1_class0","wrong_fasttrack","wrong_deny"]].round(4).to_string())

print("\nKey Insights:")
rl_r = df_sum.loc["Q-Learning","avg_reward"]
aa_r = df_sum.loc["Always Approve","avg_reward"]
rf_r = df_sum.loc["Supervised RF","avg_reward"]
print(f"  • RL vs Always-Approve reward: {rl_r:+.3f} vs {aa_r:+.3f}"
      f"  (+{rl_r-aa_r:.3f})")
print(f"  • RL vs Supervised RF reward:  {rl_r:+.3f} vs {rf_r:+.3f}"
      f"  ({rl_r-rf_r:+.3f})")
print(f"  • RL risky fast-tracks: {int(df_sum.loc['Q-Learning','wrong_fasttrack'])} "
      f"vs Always-Approve: {int(df_sum.loc['Always Approve','wrong_fasttrack'])}")

print("\nAll plots saved to /mnt/user-data/outputs/")
print("Done.")


Saved Q14_flat_reward_solution.png
Saved Q14_summary_dashboard.png

FINAL RESULTS SUMMARY
                avg_reward  accuracy     auc  f1_class0  wrong_fasttrack  wrong_deny
policy                                                                              
Random              0.2016    0.6249  0.4956     0.2401              178         578
Always Approve      2.1765    0.7613  0.5000     0.0000              705           0
Always Review       0.1230    0.7613  0.5000     0.0000                0           0
Supervised RF       0.8147    0.7640  0.5471     0.2106                4          85
Q-Learning          2.2581    0.7461  0.5212     0.1458              561         109

Key Insights:
  • RL vs Always-Approve reward: +2.258 vs +2.177  (+0.082)
  • RL vs Supervised RF reward:  +2.258 vs +0.815  (+1.443)
  • RL risky fast-tracks: 561 vs Always-Approve: 705

All plots saved to /mnt/user-data/outputs/
Done.


LOADING & PREPROCESSING DATA
Raw shape: (14767, 133)
Feature matrix: (14767, 110)
Class distribution  0=3526 (23.9%)  1=11241 (76.1%)
PCA k=15 explains 78.0% variance
Supervised RF AUC (test): 0.6845

Q12 — CLAIMS ADJUDICATION ENVIRONMENT
